# Getting started

This notebook walks through the competition data. It loads the problem sets in `data/`, builds the decoding problem from a circuit, scores a dummy decoder on the labelled set and on fresh samples, and writes the two submission files.

`data/test_set/` is a labelled set that ships with the repository. `data/kaggle_public/` and `data/kaggle_private/` are the problems you submit predictions for (no labels): download them from the Kaggle Data tab and put them in `data/` first. Run the notebook from the `notebooks/` folder.

In [4]:
import numpy as np

print(np)
print(np.__file__)
print(getattr(np, "__version__", "VERSION MISSING"))

<module 'numpy' from '/home/ubuntu-compute/miniconda3/envs/hackathon/lib/python3.12/site-packages/numpy/__init__.py'>
/home/ubuntu-compute/miniconda3/envs/hackathon/lib/python3.12/site-packages/numpy/__init__.py
1.26.4


In [1]:
import numpy as np
import stim
import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info, save_submission
print(os.getcwd())
labelled = load_kaggle_csv("../data/test_set/test_set.csv")
problems = {split: load_kaggle_csv(f"../data/kaggle_{split}/kaggle_{split}.csv") for split in ["public", "private"]}
for name, d in [("test_set", labelled), *((f"kaggle_{s}", d) for s, d in problems.items())]:
    print(f"{name}: {len(d['id'])} shots, detector events {d['detector_events'].shape}, labels included: {'observable_flips' in d}")
print("first kaggle_public ids:", ", ".join(problems["public"]["id"][:3]), "| bases:", ", ".join(np.unique(problems["public"]["basis"])))

/home/ubuntu-compute/MassStorage/git/QEC_Decoder_Hakathon_2026/notebooks


FileNotFoundError: [Errno 2] No such file or directory: '../data/kaggle_public/kaggle_public.csv'

## The circuits and the decoding problem

Each basis has its own circuit: decode X shots with the X circuit and Z shots with the Z circuit. `problem_info` turns a circuit into the check matrix $H$ (detectors × error mechanisms), the observable matrix $L$ (observables × mechanisms) and the prior probability $p$ of each mechanism, so that $D = He \bmod 2$ and $O = Le \bmod 2$.

In [8]:
circuits = {b: stim.Circuit.from_file(f"../data/test_set/{b}_circuit.stim") for b in "XZ"}
import pandas as pd
for b, circuit in circuits.items():
    info = problem_info(circuit)
    print(f"{b}: H {info['H'].shape}, L {info['L'].shape}, p from {info['p'].min():.1e} to {info['p'].max():.1e}")
    #use pandas to show the first few rows of the H and L matrices
    print(f"H:\n{pd.DataFrame(info['H']).head()}\nL:\n{pd.DataFrame(info['L']).head()}\n") 

X: H (70, 2470), L (4, 2470), p from 1.3e-04 to 2.3e-02
H:
   0     1     2     3     4     5     6     7     8     9     ...  2460  \
0     1     1     1     1     1     1     1     1     1     1  ...     0   
1     1     1     1     1     1     1     1     0     0     0  ...     0   
2     0     0     0     0     0     0     0     1     1     1  ...     0   
3     1     1     1     1     1     1     0     1     0     0  ...     0   
4     0     0     0     0     0     0     0     0     0     0  ...     0   

   2461  2462  2463  2464  2465  2466  2467  2468  2469  
0     0     0     0     0     0     0     0     0     0  
1     0     0     0     0     0     0     0     0     0  
2     0     0     0     0     0     0     0     0     0  
3     0     0     0     0     0     0     0     0     0  
4     0     0     0     0     0     0     0     0     0  

[5 rows x 2470 columns]
L:
   0     1     2     3     4     5     6     7     8     9     ...  2460  \
0     1     1     1     1     0 

## A dummy decoder

Replace `decode` with your own decoder. It takes the circuit of one basis and that basis's detector events, and returns the predicted flips of `L0` to `L3` as 0/1. It's the same function the top 10 teams hand in (see `submission_template/`).

In [2]:
def decode(circuit, detector_events):
    """Your decoder: (shots, 70) bool array -> (shots, 4) array of 0/1."""
    return np.zeros((len(detector_events), 4), dtype=np.uint8)

## Score it

The leaderboard metric is the logical error rate (LER): the fraction of predicted bits that are wrong. Lower is better. `data/test_set/` has 100,000 labelled shots per basis. It was sampled with a fixed seed, so if you tune against it, also check on freshly sampled shots.

In [3]:
for b in "XZ":
    rows = labelled["basis"] == b
    ler_labelled = logical_error_rate(decode(circuits[b], labelled["detector_events"][rows]), labelled["observable_flips"][rows])
    detector_events, observable_flips = circuits[b].compile_detector_sampler(seed=1).sample(100_000, separate_observables=True)
    ler_fresh = logical_error_rate(decode(circuits[b], detector_events), observable_flips)
    print(f"{b}: LER {ler_labelled:.4f} on data/test_set, {ler_fresh:.4f} on 100,000 fresh shots")

NameError: name 'circuits' is not defined

## Write the submissions

You submit two files: one for `kaggle_public` (scored on the Public leaderboard) and one for `kaggle_private` (scored on the Private leaderboard). `save_submission` writes the columns `id, L0, L1, L2, L3` in exactly the order Kaggle expects. Upload the files on the competition's Submit Prediction page.

In [ ]:
for split, problem in problems.items():
    predictions = np.zeros((len(problem["id"]), 4), dtype=np.uint8)
    for b in "XZ":
        rows = problem["basis"] == b
        predictions[rows] = decode(circuits[b], problem["detector_events"][rows])
    save_submission(f"../submission_{split}.csv", problem["id"], predictions)
print("".join(open("../submission_public.csv").readlines()[:3]))

id,L0,L1,L2,L3
gen_X_00000,0,0,0,0
gen_X_00001,0,0,0,0



## Next steps

- **Build your decoder** in `submission_template/decoder.py`, then run `python submission_template/run_local.py` from the repository root. It reports your LER on `data/test_set/` and times your decoder the way the code check does. Run it with `--problem-set data/kaggle_public --out submission_public.csv` and `--problem-set data/kaggle_private --out submission_private.csv` to write the two files you upload.
- **Generate more labelled data** with `python data/generate_data.py data/generated`.
- **Read the README** for the decoder ideas, the rules and how the final ranking works.